In [ ]:
import galsim
import batsim
import anacal
import numpy as np
import matplotlib.pyplot as plt

from multiprocessing import cpu_count
from parallelbar import progress_starmap
from tqdm import tqdm
from time import time

In [ ]:
def cancel_shape_noise(gal_obj, nrot):
    '''Create nrot rotated versions of the input galaxy object
    such that shape noise cancels out when averaging the shapes'''
    rotated_gals = []
    for i in range(nrot):
        rot_ang = np.pi / nrot * i
        ang = rot_ang * galsim.radians
        rotated_gals.append(gal_obj.rotate(ang))
        
    return rotated_gals

def measure_shear(gal_array, psf_array, pixel_scale, noise_variance, noise_array, detection, shear_true):
    '''Measure the shear using FPFS and return catalog'''
    fpfs_config = anacal.fpfs.FpfsConfig(
        sigma_shapelets=0.52,  # The first measurement scale (also for detection)
        sigma_shapelets2=0.45,  # The second measurement scale
    )
    catalog = anacal.fpfs.process_image(
        fpfs_config=fpfs_config,
        mag_zero=30.0,
        gal_array=gal_array,
        psf_array=psf_array,
        pixel_scale=pixel_scale,
        noise_variance=max(noise_variance, 0.23),
        noise_array=noise_array,
        detection=detection,
    )

    return catalog

def generate_images(i, nn, scale, galaxies, records, psf, ia, lens_shear, n_rot, stamp_size):
    gal = galaxies[i]
    
    # Get HLR for galaxy to create IA transform
    if records['use_bulgefit'][i]:
        hlr = records['hlr'][i][2]
    else:
        hlr = records['hlr'][i][0]

    rotated_gals = cancel_shape_noise(gal, n_rot)
    stamp = galsim.ImageF(stamp_size, stamp_size, scale=scale)
    for j in range(n_rot):
        # set drawing location on image
        row = j // int(np.sqrt(1*n_rot))
        col = j % int(np.sqrt(1*n_rot))

        # Compute the bounds for this galaxy
        xmin = col * nn + 1  # +1 because GalSim coordinates start at 1
        xmax = (col + 1) * nn
        ymin = row * nn + 1
        ymax = (row + 1) * nn

        # Decide whether or not to apply IA transform to this galaxy
        if ia:
            # Construct IA transform and simulate the galaxy with it
            IATransform = batsim.IaTransform(
                scale=scale,
                hlr=hlr,
                A=0.00136207,
                beta=0.82404653, # best first Georgiou19+
                phi = np.radians(0),
                clip_radius=5 # clip the transform at 5*hlr to prevent edge effects
            )
            gal_img = batsim.simulate_galaxy(
                ngrid=nn,
                pix_scale=scale,
                gal_obj=rotated_gals[j],
                transform_obj=IATransform,
                psf_obj=psf,
                draw_method="auto"
            )

            # Apply lensing shear directly
            gal_img = galsim.Image(gal_img, scale=scale)
            gal = galsim.InterpolatedImage(gal_img, scale=scale)
        else:
            # convolve, shift, and draw the galaxy
            gal = gal.shift(0.5*scale, 0.5*scale) # shift the galaxy to center

        # Apply lensing shear
        gal = gal.shear(g1=lens_shear, g2=0.0)

        # Convolve after both IA and lensing
        gal = galsim.Convolve([gal, psf])

        # Set the subimage in the stamp
        gal_img = gal.drawImage(nx=nn, ny=nn, scale=scale).array
        bounds = galsim.BoundsI(xmin, xmax, ymin, ymax)
        sub_image = galsim.Image(gal_img, scale=scale)
        stamp[bounds] = sub_image

    return stamp